<a href="https://colab.research.google.com/github/qja0707/OralMedicineImageDetection/blob/docs%2Fgyubeom%2Feda/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA

경구약제 객체 탐지 프로젝트 데이터 분석용 노트북

In [ ]:
import kagglehub
import os

# 환경 변수에 토큰 등록 (kaggle 개인 api 토큰. github에 저장하지마세요)
os.environ['KAGGLE_API_TOKEN'] = "KGAT_957ae188e2917ffae289bcff9a2733e5"

# Download latest version
path = kagglehub.competition_download('ai10-level1-project')

print("Path to competition files:", path)

100%|██████████| 1.79G/1.79G [00:24<00:00, 79.1MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/ai10-level1-project


In [ ]:
path = os.path.join(path, 'sprint_ai_project1_data')
print(os.listdir(path))

['test_images', 'train_annotations', 'train_images']


In [ ]:
test_images_path = os.path.join(path, 'test_images')

train_images_path = os.path.join(path, 'train_images')

train_annotations_path = os.path.join(path, 'train_annotations')

print(os.listdir(test_images_path))
print(os.listdir(train_images_path))
print(os.listdir(train_annotations_path))

['1196.png', '1298.png', '508.png', '896.png', '1025.png', '1180.png', '1328.png', '1085.png', '1170.png', '1330.png', '302.png', '798.png', '868.png', '463.png', '1003.png', '910.png', '17.png', '234.png', '1079.png', '62.png', '518.png', '1157.png', '1064.png', '46.png', '412.png', '1371.png', '1262.png', '264.png', '79.png', '114.png', '632.png', '1013.png', '941.png', '614.png', '1431.png', '1358.png', '680.png', '226.png', '1158.png', '237.png', '1361.png', '250.png', '349.png', '720.png', '205.png', '564.png', '1401.png', '897.png', '212.png', '310.png', '140.png', '826.png', '519.png', '13.png', '571.png', '1109.png', '880.png', '688.png', '864.png', '1497.png', '60.png', '542.png', '866.png', '591.png', '1156.png', '660.png', '1459.png', '815.png', '931.png', '659.png', '1435.png', '714.png', '701.png', '1115.png', '788.png', '936.png', '667.png', '493.png', '387.png', '356.png', '213.png', '422.png', '1174.png', '134.png', '299.png', '313.png', '612.png', '991.png', '507.png',

1. 이미 rotate 증강이 되어있는 것으로 보임
2. 이미지 안에 여러개의 알약이 있는 경우 알약 별로 따로 json 파일이 존재하는것으로 보임 -> 하나의 json 파일로 묶어줄 필요가 있지 않을까?

# 이미지 파일 이름 체크

In [ ]:
import re

count = 0

for root, dirs, files in os.walk(train_images_path):
    for file in files:
        count +=1
        # 확장자 제외한 파일명만 추출
        name = os.path.splitext(file)[0]

        name = name.split("_")[0]

        print(name)

        if count > 10:
            break

K-001900-016548-019607-033009
K-003351-016688-041768
K-003351-003832-021325
K-003351-018147-035206
K-002483-012081-012778-025438
K-003351-019232-029667
K-003351-019232-021325
K-003351-013900-036637
K-003351-020014-021325
K-003351-033880-035206
K-003351-016232-019232


In [ ]:
import os
import re
from collections import Counter

# 1. 정규표현식 및 초기화
code_extractor = re.compile(r'(?<=K-)\d{6}|(?<=-)\d{6}')
pill_counts = Counter() # 이 자체가 {"code": n} 형태의 맵이 됩니다.

# 2. 순회 및 카운트
for root, dirs, files in os.walk(train_images_path):
    for file in files:
        if file.startswith('.'): continue

        # 파일명 정제 (확장자 제거 후 _ 앞부분만)
        clean_name = os.path.splitext(file)[0].split("_")[0]

        # 코드 추출 및 맵 업데이트
        codes = code_extractor.findall(clean_name)
        pill_counts.update(codes) # 리스트를 넣으면 자동으로 {"코드": 개수} 업데이트

# 3. 전체 합계 및 통계 계산
total_pill_count = sum(pill_counts.values())
sorted_pill_map = dict(sorted(pill_counts.items()))

print(f"📊 [알약 데이터셋 전체 통계]")
print(f" - 총 알약 등장 횟수(Total Sum): {total_pill_count}개")
print(f" - 유니크 알약 종류: {len(sorted_pill_map)}종")
print("-" * 45)

# 4. 전체 데이터 출력 (코드 | 빈도 | 퍼센트)
print(f"{'코드':<10} | {'빈도':<8} | {'비중(%)':<10}")
print("-" * 45)

for code, count in sorted_pill_map.items():
    percentage = (count / total_pill_count) * 100
    print(f"{code:<10} | {count:<10} | {percentage:>6.2f}%")


📊 [알약 데이터셋 전체 통계]
 - 총 알약 등장 횟수(Total Sum): 771개
 - 유니크 알약 종류: 56종
---------------------------------------------
코드         | 빈도       | 비중(%)     
---------------------------------------------
001900     | 15         |   1.95%
002483     | 9          |   1.17%
003351     | 157        |  20.36%
003483     | 45         |   5.84%
003544     | 6          |   0.78%
003743     | 3          |   0.39%
003832     | 20         |   2.59%
004543     | 6          |   0.78%
012081     | 3          |   0.39%
012247     | 3          |   0.39%
012778     | 6          |   0.78%
013395     | 3          |   0.39%
013900     | 15         |   1.95%
016232     | 21         |   2.72%
016262     | 23         |   2.98%
016548     | 18         |   2.33%
016551     | 3          |   0.39%
016688     | 8          |   1.04%
018147     | 15         |   1.95%
018357     | 13         |   1.69%
019232     | 17         |   2.20%
019552     | 3          |   0.39%
019607     | 6          |   0.78%
019861     | 6          

# 💊 경구약제 객체 탐지 모델 추천 (High Accuracy)

이미지 하나에 여러 종류의 알약이 포함되어 있고, 실시간성보다는 **정확도**와 **작은 객체(Small Object) 탐지**가 중요한 프로젝트에 최적화된 SOTA 모델 추천입니다.

### 1. RT-DETR (Real-Time DEtection TRansformer)
*   **개요**: Transformer 구조를 객체 탐지에 효율적으로 접목시킨 최신 모델입니다.
*   **강점**:
    *   **객체 간 관계 파악**: 알약이 서로 밀집해 있거나 겹쳐 있는 상황에서 YOLO보다 객체 하나하나를 더 정확하게 구분합니다.
    *   **정교한 박싱**: NMS(중복 박스 제거) 과정이 필요 없는 구조적 특성상, 박스의 위치가 매우 정확합니다.
*   **추천 상황**: 알약의 배치가 복잡하고, 비슷한 모양의 알약이 모여 있는 데이터를 다룰 때 가장 추천합니다.

### 2. DINO (DETR with Improved Denoising Boxes)
*   **개요**: 현재 객체 탐지 분야 리더보드(COCO Dataset 등)에서 최상위권의 성능을 보여주는 모델입니다.
*   **강점**:
    *   **절대적 정확도**: 현존 모델 중 mAP(평균 정밀도) 점수가 가장 높습니다.
    *   **미세 특징 추출**: 알약 표면의 각인(문자), 분할선 등 아주 미세한 특징을 잡아내는 능력이 탁월합니다.
*   **추천 상황**: 추론 시간이 조금 걸리더라도 **무조건 최고의 성능**을 뽑아내야 하는 연구용/대회용 프로젝트에 적합합니다.

### 3. YOLO11x (Extra Large 버전)
*   **개요**: 2024년 말 출시된 YOLO 시리즈의 최신 모델 중 가장 큰 파라미터를 가진 버전입니다.
*   **강점**:
    *   **높은 범용성**: `ultralytics` 라이브러리를 사용해 학습 및 배포가 매우 간편하며 커뮤니티 지원이 활발합니다.
    *   **개선된 작은 객체 탐지**: 이전 버전(v8, v10)보다 작은 물체를 찾아내는 알고리즘이 크게 강화되었습니다.
*   **추천 상황**: 빠른 구현과 학습이 필요하면서도, 높은 성능을 동시에 챙기고 싶을 때 추천합니다.

---

## 🚀 성능 향상을 위한 핵심 전략

알약 탐지 데이터셋의 특성상 모델 선정만큼 중요한 설정들입니다.

1. **입력 해상도(Resolution) 확장**: 알약의 각인을 식별하기 위해 모델 입력 크기를 크게 설정하세요. (예: `imgsz=1024`)
2. **SAHI (Slicing Aided Hyper Inference)**: 이미지 내 알약이 너무 작다면, 전체 이미지를 쪼개서(Slicing) 부분별로 탐지한 뒤 합치는 기법을 적용해 보세요. 작은 객체 인식률이 비약적으로 향상됩니다.
3. **Augmentation**: 알약의 색상, 조명, 각도 변화에 대응할 수 있도록 `Color Jitter`나 `Rotation` 증강 기법을 적극 활용하세요.
